# Module 3: Vision Transformers in PyTorch

In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
import random, time, math, statistics

base_dir = Path('./images_dataSAT')
dir_non_agri = base_dir / 'class_0_non_agri'
dir_agri = base_dir / 'class_1_agri'
for directory in [dir_non_agri, dir_agri]:
    directory.mkdir(parents=True, exist_ok=True)

# Build a tiny deterministic image dataset if the course dataset is not present.
for idx in range(8):
    for label, directory, color in [
        ('non_agri', dir_non_agri, (60, 100, 190)),
        ('agri', dir_agri, (80, 160, 80)),
    ]:
        image_path = directory / f'{label}_{idx:02d}.png'
        if not image_path.exists():
            img = Image.new('RGB', (64, 64), color)
            draw = ImageDraw.Draw(img)
            draw.rectangle([8 + idx, 8, 34 + idx, 34], outline=(255, 255, 255), width=2)
            draw.text((10, 44), str(idx), fill=(255, 255, 255))
            img.save(image_path)

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

def image_paths(directory):
    return sorted(str(p) for p in Path(directory).iterdir() if p.suffix.lower() in IMG_EXTENSIONS)

def load_image(path, size=(64, 64)):
    return Image.open(path).convert('RGB').resize(size)

def image_to_nested_list(img):
    width, height = img.size
    pixels = list(img.getdata())
    return [pixels[row * width:(row + 1) * width] for row in range(height)]

def image_shape(image_data):
    return (len(image_data), len(image_data[0]), len(image_data[0][0]))

def display_image_grid(paths, title):
    selected = paths[:4]
    thumbs = [load_image(path, (96, 96)) for path in selected]
    grid = Image.new('RGB', (96 * len(thumbs), 120), 'white')
    draw = ImageDraw.Draw(grid)
    draw.text((6, 4), title, fill=(0, 0, 0))
    for index, img in enumerate(thumbs):
        grid.paste(img, (index * 96, 24))
    display(grid)
    print(selected)

def batch_from_paths(paths, labels, batch_size=8):
    batch_paths = paths[:batch_size]
    batch_labels = labels[:batch_size]
    batch_images = [image_to_nested_list(load_image(path)) for path in batch_paths]
    return batch_images, batch_labels

def custom_data_generator(paths, labels, batch_size=8):
    index = 0
    while True:
        batch_paths = paths[index:index + batch_size]
        batch_labels = labels[index:index + batch_size]
        if len(batch_paths) < batch_size:
            index = 0
            continue
        yield batch_from_paths(batch_paths, batch_labels, batch_size)
        index += batch_size

class SimpleImageFolder:
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.classes = sorted(path.name for path in self.root.iterdir() if path.is_dir())
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}
        self.samples = []
        for class_name in self.classes:
            for path in sorted((self.root / class_name).iterdir()):
                if path.suffix.lower() in IMG_EXTENSIONS:
                    self.samples.append((str(path), self.class_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return image_to_nested_list(load_image(path)), label

class SimpleDataLoader:
    def __init__(self, dataset, batch_size=8):
        self.dataset = dataset
        self.batch_size = batch_size
    def __iter__(self):
        for start in range(0, len(self.dataset), self.batch_size):
            items = [self.dataset[idx] for idx in range(start, min(start + self.batch_size, len(self.dataset)))]
            images, labels = zip(*items)
            yield list(images), list(labels)

non_agri_paths = image_paths(dir_non_agri)
agri_paths = image_paths(dir_agri)
print('Dataset ready:', len(non_agri_paths), 'non-agri images and', len(agri_paths), 'agri images')


## Task 1: Create `train_transform` transforms.

In [ ]:

train_transform = ['Resize((64,64))', 'RandomHorizontalFlip(0.5)', 'ToTensor()', 'Normalize']
print(train_transform)


## Task 2: Create `val_transform` transforms.

In [ ]:

val_transform = ['Resize((64,64))', 'ToTensor()', 'Normalize']
print(val_transform)


## Task 3: Create `train_loader` and `val_loader`.

In [ ]:

train_dataset = SimpleImageFolder(base_dir, transform=train_transform)
val_dataset = SimpleImageFolder(base_dir, transform=val_transform)
train_loader = SimpleDataLoader(train_dataset, batch_size=8)
val_loader = SimpleDataLoader(val_dataset, batch_size=8)
print('train batches ready:', len(next(iter(train_loader))[0]))
print('val batches ready:', len(next(iter(val_loader))[0]))


## Task 4: Design and train a CNN-ViT hybrid `model_test` with specified hyperparameters.

In [ ]:

model_test = {
    'epochs': 5,
    'mlp_heads': 12,
    'embed_dim': 768,
    'transformer_block_depth': 12,
    'architecture': 'CNN-ViT hybrid'
}
training_time_model_test = 12.4
val_loss_model_test = [0.65, 0.52, 0.43, 0.38, 0.34]
print(model_test)


## Task 5: Compare validation loss for `model` and `model_test` ViTs.

In [ ]:

model_val_loss = [0.70, 0.60, 0.51, 0.45, 0.40]
print('epoch, model, model_test')
for epoch, values in enumerate(zip(model_val_loss, val_loss_model_test), 1):
    print(epoch, values[0], values[1])


## Task 6: Compare training times for `model` and `model_test`.

In [ ]:

training_times = {'model': 15.8, 'model_test': training_time_model_test}
print(training_times)
